# Semi-Supervised Seeded Clustering Calibration

This notebook calibrates the incremental seeded clustering algorithm across model/seed/threshold grids.
Each completed run is appended to a live results table for immediate feedback.

In [1]:
import os
import pandas as pd
import numpy as np
from IPython.display import display

import config
import model_utils_seeded as model_utils
import model_utils_shared
import warnings

warnings.simplefilter(action='ignore', category=FutureWarning)

# Reproducibility
model_utils_shared.setup_reproducibility(config.SEED)

/home/rass/Desktop/SocialScience-ConceptIntegration/myenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


42

## 1. Load Data

In [2]:
pos_path = config.DATA_PATHS['pos_pairs']
neg_path = config.DATA_PATHS['neg_pairs']

df_pos = pd.read_csv(pos_path)
df_neg = pd.read_csv(neg_path)
df_train = pd.concat([df_pos, df_neg], ignore_index=True)

df_pos['term1'] = df_pos['term1'].astype(str)
df_pos['term2'] = df_pos['term2'].astype(str)
df_neg['term1'] = df_neg['term1'].astype(str)
df_neg['term2'] = df_neg['term2'].astype(str)
df_train['term1'] = df_train['term1'].astype(str)
df_train['term2'] = df_train['term2'].astype(str)

print(f"Positive samples: {len(df_pos)}")
print(f"Negative samples: {len(df_neg)}")
print(f"Total rows: {len(df_train)}")
df_train.head()

Positive samples: 3587
Negative samples: 2884723
Total rows: 2888310


,term1,term2,concept_uri,label,shortest_path,concept1_uri,concept2_uri
0,HEADS OF STATE,MONARCHS,https://elsst.cessda.eu/id/5/01528e98-3454-4a4...,1,0,NaN,NaN
1,HEADS OF STATE,PRESIDENTS,https://elsst.cessda.eu/id/5/01528e98-3454-4a4...,1,0,NaN,NaN
2,HEADS OF STATE,SOVEREIGNS,https://elsst.cessda.eu/id/5/01528e98-3454-4a4...,1,0,NaN,NaN
3,MONARCHS,PRESIDENTS,https://elsst.cessda.eu/id/5/01528e98-3454-4a4...,1,0,NaN,NaN
4,MONARCHS,SOVEREIGNS,https://elsst.cessda.eu/id/5/01528e98-3454-4a4...,1,0,NaN,NaN


## 2. Seeded Clustering Calibration Loop

The table below updates after each completed run (model, seed count, threshold, run ID).

In [3]:
flat_terms = pd.unique(df_train[['term1', 'term2']].values.ravel('K'))
unique_terms = [str(term) for term in flat_terms if pd.notna(term)]
term_to_idx = {term: i for i, term in enumerate(unique_terms)}
pos_idx1, pos_idx2 = model_utils.prepare_pair_indices(df_pos, term_to_idx)
neg_idx1, neg_idx2 = model_utils.prepare_pair_indices(df_neg, term_to_idx)

seed_counts = [10, 25, 50, 100, 250, 500]
run_states = [config.SEED + i for i in range(5)]

results_cols = [
    'Model', 'n_seeds', 'Threshold', 'Run_ID',
    'F1', 'Precision', 'Recall', 'Pos_Acc', 'Neg_Acc', 'Num_Clusters'
]
results_df = pd.DataFrame(columns=results_cols)
display_handle = display(results_df.head(0), display_id=True)
# Keep the live table small for faster UI updates.

results_dir = config.DATA_PATHS['results_dir']
os.makedirs(results_dir, exist_ok=True)
output_path = os.path.join(results_dir, 'seeded_calibration_results.csv')

print(f"{'Model':<28} | {'Seeds':<5} | {'Thresh':<6} | {'Run':<3} | {'F1':<7} | {'Prec':<7} | {'Rec':<7} | {'PosAcc':<7} | {'NegAcc':<7} | {'Clusters':<8}")
print('-' * 120)

for model_name, model_info in config.MODELS.items():
    model_type = model_info.get('type', config.TYPE_SENTENCE)
    print(f"Loading model: {model_name}")
    model = model_utils_shared.load_model(model_name, model_type)
    if model is None:
        continue

    print(f"Encoding {len(unique_terms)} terms...")
    embeddings = model.encode(
        unique_terms,
        convert_to_tensor=False,
        show_progress_bar=True,
        batch_size=config.BATCH_SIZE
    )
    embeddings = model_utils._l2_normalize(embeddings)
    n_terms = len(unique_terms)

    for n_seeds in seed_counts:
        print(f"Calibrating seed count: {n_seeds}")
        for run_id, random_state in enumerate(run_states, start=1):
            rng = np.random.default_rng(random_state)
            order = rng.permutation(n_terms)
            seed_indices = model_utils.sample_distinct_seeds(
                df_train,
                n_seeds,
                terms=unique_terms,
                random_state=random_state
            )

            for threshold in config.THRESHOLDS:
                cluster_labels = model_utils._run_seeded_clustering_normalized(
                    embeddings,
                    seed_indices,
                    threshold,
                    order=order
                )

                precision, recall, f1, pos_acc, neg_acc = model_utils.evaluate_clusters_from_indices(
                    cluster_labels,
                    pos_idx1,
                    pos_idx2,
                    neg_idx1,
                    neg_idx2
                )

                n_clusters = int(cluster_labels.max()) + 1 if len(cluster_labels) else 0
                row = {
                    'Model': model_name,
                    'n_seeds': n_seeds,
                    'Threshold': float(threshold),
                    'Run_ID': run_id,
                    'F1': f1,
                    'Precision': precision,
                    'Recall': recall,
                    'Pos_Acc': pos_acc,
                    'Neg_Acc': neg_acc,
                    'Num_Clusters': n_clusters
                }
                results_df.loc[len(results_df)] = row

                display_handle.update(results_df.tail(1))
                print(
                    f"{model_name:<28} | {n_seeds:<5} | {threshold:>6.2f} | {run_id:<3} | "
                    f"{f1:>7.4f} | {precision:>7.4f} | {recall:>7.4f} | "
                    f"{pos_acc:>7.4f} | {neg_acc:>7.4f} | {n_clusters:>8}"
                )

results_df.to_csv(output_path, index=False)
print(f"Saved results to {output_path}")


,Model,n_seeds,Threshold,Run_ID,F1,Precision,Recall,Pos_Acc,Neg_Acc,Num_Clusters
3149,dwulff/mpnet-personality,10,0.99,5,0.0,0.0,0.0,0.0,1.0,4270


Model                        | Seeds | Thresh | Run | F1      | Prec    | Rec     | PosAcc  | NegAcc  | Clusters
------------------------------------------------------------------------------------------------------------------------
Loading model: all-mpnet-base-v2
Loading Model (all-mpnet-base-v2)...
Encoding 4270 terms...


Batches: 100%|██████████| 267/267 [00:17<00:00, 15.26it/s]


Calibrating seed count: 10
all-mpnet-base-v2            | 10    |   0.10 | 1   |  0.0128 |  0.0065 |  0.6730 |  0.6730 |  0.8713 |       11
all-mpnet-base-v2            | 10    |   0.11 | 1   |  0.0128 |  0.0065 |  0.6730 |  0.6730 |  0.8713 |       11
all-mpnet-base-v2            | 10    |   0.12 | 1   |  0.0130 |  0.0066 |  0.6646 |  0.6646 |  0.8748 |       11
all-mpnet-base-v2            | 10    |   0.13 | 1   |  0.0137 |  0.0069 |  0.6599 |  0.6599 |  0.8823 |       12
all-mpnet-base-v2            | 10    |   0.14 | 1   |  0.0137 |  0.0069 |  0.6599 |  0.6599 |  0.8823 |       12
all-mpnet-base-v2            | 10    |   0.15 | 1   |  0.0137 |  0.0069 |  0.6599 |  0.6599 |  0.8823 |       12
all-mpnet-base-v2            | 10    |   0.16 | 1   |  0.0149 |  0.0075 |  0.6800 |  0.6800 |  0.8885 |       14
all-mpnet-base-v2            | 10    |   0.17 | 1   |  0.0149 |  0.0075 |  0.6800 |  0.6800 |  0.8885 |       14
all-mpnet-base-v2            | 10    |   0.18 | 1   |  0.0163 |  0.00

Batches: 100%|██████████| 267/267 [00:16<00:00, 16.15it/s]


Calibrating seed count: 10
dwulff/mpnet-personality     | 10    |   0.10 | 1   |  0.0087 |  0.0044 |  0.7137 |  0.7137 |  0.7981 |       10
dwulff/mpnet-personality     | 10    |   0.11 | 1   |  0.0087 |  0.0044 |  0.7137 |  0.7137 |  0.7981 |       10
dwulff/mpnet-personality     | 10    |   0.12 | 1   |  0.0087 |  0.0044 |  0.7137 |  0.7137 |  0.7981 |       10
dwulff/mpnet-personality     | 10    |   0.13 | 1   |  0.0087 |  0.0044 |  0.7137 |  0.7137 |  0.7981 |       10
dwulff/mpnet-personality     | 10    |   0.14 | 1   |  0.0087 |  0.0044 |  0.7137 |  0.7137 |  0.7981 |       10
dwulff/mpnet-personality     | 10    |   0.15 | 1   |  0.0087 |  0.0044 |  0.7137 |  0.7137 |  0.7981 |       10
dwulff/mpnet-personality     | 10    |   0.16 | 1   |  0.0087 |  0.0044 |  0.7081 |  0.7081 |  0.7987 |       11
dwulff/mpnet-personality     | 10    |   0.17 | 1   |  0.0088 |  0.0044 |  0.7009 |  0.7009 |  0.8036 |       12
dwulff/mpnet-personality     | 10    |   0.18 | 1   |  0.0088 |  0.00

KeyboardInterrupt: 